In [8]:
! pip install pdfplumber==0.11.0 \
langchain==0.2.16 \
chromadb==0.5.5 \
sentence-transformers==3.0.1

In [9]:
import os
import pandas as pd
from getpass import getpass
import pdfplumber
import chromadb
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)
from chromadb.utils import embedding_functions
import re

### Ingestion of PDFs

In [10]:
def extract_year(filename):
    # pattern for 2023-24 or 2022_23
    match = re.search(r'(20\d{2})[-_](\d{2})', filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    # pattern for 202021 or 202122
    match = re.search(r'(20\d{2})(\d{2})', filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    return "unknown"

def clean_text(text):
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def load_pdfs(pdf_dir):
  files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
  pdfs = []

  print(f"Found {len(files)} pdfs.")

  for file in files:
      filepath = os.path.join(pdf_dir, file)
      year = extract_year(file)
      text = ""

      with pdfplumber.open(filepath) as pdf:
          for page in pdf.pages:
            content = page.extract_text()
            if content:
              text += content + "\n"

      text = clean_text(text)

      pdfs.append({
        "file": file,
        "year": year,
        "filepath": filepath,
        "text": text
      })

      print(f"Loaded {file} | {year} | {len(text.split())} words")

  return pdfs

In [ ]:
documents = load_pdfs("/content/")

Found 5 pdfs.
Loaded GCPL-Annual-and-Integrated-Report-2023-24.pdf | 2023-24 | 189756 words
Loaded GCPL-Annual-and-Integrated-Report-2024-25.pdf | 2024-25 | 195361 words
Loaded GCPL_Annual_Report_202122.pdf | 2021-22 | 138352 words
Loaded GCPL_Annual_Report_202021.pdf | 2020-21 | 129183 words
Loaded GCPL_Annual_Report_2022_23.pdf | 2022-23 | 142538 words


### Chunking

In [11]:
def chunk_fixed(documents):
  splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")
  chunks = []

  for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
      chunks.append({
        "id": f"{doc['year']}_fixed_{i}",
        "text": chunk,
        "source": doc["file"],
        "year": doc["year"],
        "strategy": "fixed"
     })
  return chunks

In [ ]:
fixed_chunks = chunk_fixed(documents)
print("Fixed chunks:", len(fixed_chunks))

Fixed chunks: 11346


In [12]:
def chunk_recursive(documents):
  splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=120, separators=["\n\n", "\n• ", "\n- ", "\n", ". ", " ", ""])
  chunks = []

  for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
      chunk = chunk.strip()
      if len(chunk) < 80:
        continue
      chunks.append({
        "id": f"{doc['year']}_recursive_{i}",
        "text": chunk,
        "source": doc["file"],
        "year": doc["year"],
        "strategy": "recursive"
     })
  return chunks

In [ ]:
recursive_chunks = chunk_recursive(documents)
print("Recursive chunks:", len(recursive_chunks))

Recursive chunks: 9157


### Embedding

In [13]:
def get_local_ef():
    return embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

def get_local_ef_v2():
    return embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="multi-qa-MiniLM-L6-cos-v1"
    )

def build_vector_store(chroma_dir, chunks, collection_name, embedding_fn):
    client = chromadb.PersistentClient(path=chroma_dir)
    # recreate collection
    try:
        client.delete_collection(collection_name)
    except:
        pass
    collection = client.create_collection(name=collection_name, embedding_function=embedding_fn)
    collection.add(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        metadatas=[
            {
                "source": c["source"],
                "year": c["year"],
                "strategy": c["strategy"]
            }
            for c in chunks
        ]
    )
    print(f"{collection_name}: {collection.count()} vectors stored")
    return collection

In [ ]:
embedding_fn = get_local_ef()
chroma_dir = "/content/chroma_database"

build_vector_store(chroma_dir, fixed_chunks, "fixed__local", embedding_fn)
build_vector_store(chroma_dir, recursive_chunks, "recursive__local", embedding_fn)

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


fixed__local: 11346 vectors stored


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


recursive__local: 9157 vectors stored


Collection(id=85c09118-ab64-4bc0-8efd-7855c91e6bde, name=recursive__local)

In [ ]:
embedding_fn_v2 = get_local_ef_v2()
chroma_dir_v2 = "/content/chroma_database_v2"

build_vector_store(chroma_dir_v2, fixed_chunks, "fixed__local", embedding_fn_v2)
build_vector_store(chroma_dir_v2, recursive_chunks, "recursive__local", embedding_fn_v2)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


fixed__local: 11346 vectors stored


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


recursive__local: 9157 vectors stored


Collection(id=acfca378-531d-4eb1-8511-78c3d8ee034b, name=recursive__local)

In [14]:
!zip -r /content/chroma_database.zip /content/chroma_database

	zip warning: name not matched: /content/chroma_database

zip error: Nothing to do! (try: zip -r /content/chroma_database.zip . -i /content/chroma_database)


In [15]:
!zip -r /content/chroma_database_v2.zip /content/chroma_database_v2

	zip warning: name not matched: /content/chroma_database_v2

zip error: Nothing to do! (try: zip -r /content/chroma_database_v2.zip . -i /content/chroma_database_v2)


In [16]:
!unzip /content/chroma_database.zip -d /content/

Archive:  /content/chroma_database.zip
replace /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/data_level0.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/data_level0.bin  
replace /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/length.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/length.bin  
  inflating: /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/index_metadata.pickle  
  inflating: /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/header.bin  
  inflating: /content/content/chroma_database/d405cc38-c518-4da7-af78-c156b33972a4/link_lists.bin  
  inflating: /content/content/chroma_database/chroma.sqlite3  
  inflating: /content/content/chroma_database/3464a162-f158-4ea7-868d-20a096855a08/data_level0.bin  
  inflating: /content/content/ch

In [17]:
!unzip /content/chroma_database_v2.zip -d /content/

Archive:  /content/chroma_database_v2.zip
replace /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/data_level0.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/data_level0.bin  
  inflating: /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/length.bin  
  inflating: /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/index_metadata.pickle  
  inflating: /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/header.bin  
  inflating: /content/content/chroma_database_v2/78dbde41-4a35-4bef-ad81-6c7d143a4610/link_lists.bin  
  inflating: /content/content/chroma_database_v2/55af7398-3cba-4e2a-9fea-3323296955e7/data_level0.bin  
  inflating: /content/content/chroma_database_v2/55af7398-3cba-4e2a-9fea-3323296955e7/length.bin  
  inflating: /content/content/chroma_database_v2/55af7398-3cba-4e2a-9fea-3323296955e7/index_metadata.p

### Retrieving the Information

In [18]:
def retrieve(chroma_dir, collection_name, query, k=5):
    client = chromadb.PersistentClient(path=chroma_dir)
    collection = client.get_collection(
        name=collection_name,
        embedding_function=get_local_ef()
    )
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    return results

In [19]:
queries = [
    "What supply chain risks did GCPL highlight?",
    "What sustainability initiatives are mentioned in the report?",
    "What were the major financial highlights of GCPL?",
    "What challenges does GCPL mention in raw material sourcing?",
    "What are GCPL's key product categories?"
]

In [27]:
chroma_dir_v1 = "/content/content/chroma_database"
chroma_dir_v2 = "/content/content/chroma_database_v2"

CONFIGS = {
    "fixed__local":          (chroma_dir_v1, "fixed | MiniLM-v2"),
    "recursive__local":      (chroma_dir_v1, "recursive | MiniLM-v2"),
    "fixed__local_v2":       (chroma_dir_v2, "fixed | multi-qa-MiniLM"),
    "recursive__local_v2":   (chroma_dir_v2, "recursive | multi-qa-MiniLM"),
}

In [28]:
for col, (cdir, label) in CONFIGS.items():
    print("\n\n===================")
    print("Collection:", col)
    for q in queries:
        print("\nQuery:", q)
        results = retrieve(cdir, col.replace("_v2", ""), q)
        for doc in results["documents"][0][:2]:
            print("-", doc[:150])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




Collection: fixed__local

Query: What supply chain risks did GCPL highlight?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


- supply chain of materials and services from chain engagement, GCPL has compliance by critical suppliers
management diverse suppliers. Some of the coll
- the key risks that can by procurement spends for
impact the company’s their ESG performance. To
supply chain are ESG drive supplier engagement
complia

Query: What sustainability initiatives are mentioned in the report?
- Reporting Standards)
This engagement was conducted by a multidisciplinary team including assurance practitioners, engineers and
environmental and soci
- These initiatives have led to a reduction
in manpower requirements and significant
cost savings.
110
Our processes include:
• Extensive meetings with 

Query: What were the major financial highlights of GCPL?
- Our R&D teams lead new product development
across the geographies we operate in
4
Content of Scope and Forward Looking
the report Boundary of the Stat
- with high potential, GCPL aims to
compensation and well-being. It
performance. It is designed
provide tota

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


- supply chain of materials and services from chain engagement, GCPL has compliance by critical suppliers
management diverse suppliers. Some of the coll
- • Extending leadership in our core categories
and geographies
• Accelerating innovation and building
purposeful brands
• Leveraging digital
• Enhancin

Query: What sustainability initiatives are mentioned in the report?
- • Conducting extensive meetings with • Implementing a robust governance
various stakeholders to align priorities, framework to monitor, assess, and
bu
- Directors. The Committee’s execution of
prime responsibilities are as such projects or • Review progress
under: programs; against Sustainability
goals

Query: What were the major financial highlights of GCPL?
- endorsement
Through integrated reporting, we aim to This report is for GCPL, including GCPL
and assurance
share an overview of our financial and non- 
- • The linkage between material concerns, consistent, and investor-relevant view of our as of the reporting

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional arg

- supply chain of materials and services from chain engagement, GCPL has compliance by critical suppliers
management diverse suppliers. Some of the coll
- the key risks that can by procurement spends for
impact the company’s their ESG performance. To
supply chain are ESG drive supplier engagement
complia

Query: What sustainability initiatives are mentioned in the report?
- usage demands. We achieved zero waste indices has improved, returning to the (2) health and well-being that protects,
to landfill and water positivity
- These initiatives have led to a reduction
in manpower requirements and significant
cost savings.
110
Our processes include:
• Extensive meetings with 

Query: What were the major financial highlights of GCPL?
- • The linkage between material concerns, consistent, and investor-relevant view of our as of the reporting date and are not
strategic responses, opera
- 4
Content of Scope and Management
the report boundary Committee
endorsement and
Through integrated reporti

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


- supply chain of materials and services from chain engagement, GCPL has compliance by critical suppliers
management diverse suppliers. Some of the coll
- services from diverse and quantitative data, and
suppliers. Some of evaluated 70% of suppliers
the key risks that can by procurement spends for
impact

Query: What sustainability initiatives are mentioned in the report?
- budget and initiated medium to long- measures and intermittent operations.
term livelihood recovery programmes
to support 9,000 nano entrepreneurs.
Pi
- usage demands. We achieved zero waste indices has improved, returning to the (2) health and well-being that protects,
to landfill and water positivity

Query: What were the major financial highlights of GCPL?
- endorsement
Through integrated reporting, we aim to This report is for GCPL, including GCPL
and assurance
share an overview of our financial and non- 
- We aim to provide an overview of our This report covers GCPL, including its The Integrated Annual Report i

### Evaluation of Retrieval Task

In [29]:
def precision_at_k(results, keywords):
    docs     = results["documents"][0]
    relevant = 0
    for d in docs:
        if any(k.lower() in d.lower() for k in keywords):
            relevant += 1
    return relevant / len(docs)

In [30]:
def recall_at_k(results, keywords):
    docs  = results["documents"][0]
    found = 0
    for k in keywords:
        if any(k.lower() in d.lower() for d in docs):
            found += 1
    return found / len(keywords)

In [45]:
def qualitative_score(generated, reference):
    gen = str(generated).lower().strip()
    if not gen or gen == "nan":
        return 0
    ref_words = [w for w in reference.lower().split()[:10]
                 if len(w) > 3]           # skip stop words like "the", "in", "a"
    if not ref_words:
        return 0
    hits = sum(w in gen for w in ref_words)
    return 2 if hits >= 3 else (1 if hits >= 1 else 0)

In [31]:
test_set = [

    # ── FY 2020-21 ────────────────────────────────────────────────────────────

    {
        "id": "Q01",
        "query": "What major shift in consumer behavior did GCPL observe across its markets during FY 2020-21?",
        "reference_answer": "GCPL witnessed a sharp shift in shopper demand for health and hygiene products across its global markets during FY 2020-21 due to the COVID-19 pandemic.",
        "keywords": ["health", "hygiene", "COVID", "pandemic", "demand", "shift"],
        "relevant_years": ["2020-21"],
        "type": "inferential",
        "difficulty": "easy",
    },
    {
        "id": "Q02",
        "query": "When did GCPL conduct the extensive materiality exercise that formed the basis for its reporting methodology in subsequent years?",
        "reference_answer": "GCPL conducted an extensive materiality exercise in FY 2020-21, involving an external partner to understand the relationship between material issues and business risks.",
        "keywords": ["materiality", "2020-21", "external partner", "business risks", "reporting"],
        "relevant_years": ["2020-21"],
        "type": "factual",
        "difficulty": "hard",   # niche — unlikely to surface without good chunking
    },

    # ── FY 2021-22 ────────────────────────────────────────────────────────────

    {
        "id": "Q03",
        "query": "What was the percentage of representation for local ethnic talent in GCPL's Africa senior leadership roles during FY 2021-22?",
        "reference_answer": "Representation of local ethnic talent in senior leadership roles in Africa was 33% in FY 2021-22.",
        "keywords": ["33%", "Africa", "local", "ethnic", "senior leadership"],
        "relevant_years": ["2021-22"],
        "type": "factual_numeric",
        "difficulty": "hard",   # very specific stat, tests precise retrieval
    },
    {
        "id": "Q04",
        "query": "How many financial years had GCPL maintained a status of sending zero waste to landfills by the end of FY 2022-23?",
        "reference_answer": "By the end of FY 2022-23, GCPL had maintained zero waste to landfills from its manufacturing units for the last 4 financial years, which includes FY 2021-22.",
        "keywords": ["zero waste", "landfill", "4 years", "manufacturing"],
        "relevant_years": ["2021-22", "2022-23"],
        "type": "factual_numeric",
        "difficulty": "medium",
    },

    # ── FY 2022-23 ────────────────────────────────────────────────────────────

    {
        "id": "Q05",
        "query": "How did GCPL's consolidated revenue and volume grow in FY 2022-23, and how did the Chairperson characterize the year's performance?",
        "reference_answer": "In FY 2022-23, value growth was 8% and volume growth was -1%. The Chairperson characterized it as a 'moderate year of performance' but a 'strong year of strategic transformation'.",
        "keywords": ["8%", "volume", "-1%", "Chairperson", "strategic transformation", "moderate"],
        "relevant_years": ["2022-23"],
        "type": "factual_numeric",
        "difficulty": "medium",
    },
    {
        "id": "Q06",
        "query": "What specific strategic change was implemented in the Indonesia business to simplify sales and distribution during FY 2022-23?",
        "reference_answer": "GCPL moved from a direct distribution model to a distributor-led model in Indonesia to simplify sales and reduce operational complexity in General Trade.",
        "keywords": ["Indonesia", "distributor", "distribution", "General Trade", "direct"],
        "relevant_years": ["2022-23"],
        "type": "factual",
        "difficulty": "medium",
    },

    # ── FY 2023-24 ────────────────────────────────────────────────────────────

    {
        "id": "Q07",
        "query": "What was the total R&D expenditure for GCPL in FY 2023-24, and what percentage of total sales turnover did it represent?",
        "reference_answer": "The total R&D expenditure was ₹26.11 crore, which represented 0.32% of total sales turnover in FY 2023-24.",
        "keywords": ["R&D", "26.11", "0.32%", "sales turnover", "expenditure"],
        "relevant_years": ["2023-24"],
        "type": "factual_numeric",
        "difficulty": "hard",   # very specific figures — good stress test
    },
    {
        "id": "Q08",
        "query": "How many days was the number of days of accounts payables for GCPL in FY 2023-24?",
        "reference_answer": "The number of days for accounts payables was 56 days in FY 2023-24.",
        "keywords": ["accounts payable", "56 days", "payables"],
        "relevant_years": ["2023-24"],
        "type": "factual_numeric",
        "difficulty": "hard",   # buried in financial tables — tests chunking quality
    },

    # ── FY 2024-25 ────────────────────────────────────────────────────────────

    {
        "id": "Q09",
        "query": "What was the financial impact of the Greenfield facility investment at Chengalpattu and what is its strategic designation?",
        "reference_answer": "GCPL invested ₹500 crore in the Chengalpattu facility, designated as a 'lighthouse' factory — digital-first, efficient, and a model for inclusive hiring.",
        "keywords": ["500 crore", "Chengalpattu", "lighthouse", "digital", "inclusive"],
        "relevant_years": ["2024-25"],
        "type": "factual",
        "difficulty": "medium",
    },
    {
        "id": "Q10",
        "query": "What progress did GCPL make toward its Greener Products goal as part of its sustainability targets for FY 2024-25?",
        "reference_answer": "GCPL achieved a place in the Dow Jones Sustainability Index (DJSI), driven by meeting targets for post-consumer recycled (PCR) plastic and overall plastic reduction ahead of timelines.",
        "keywords": ["DJSI", "Dow Jones", "PCR", "plastic reduction", "greener", "recycled"],
        "relevant_years": ["2024-25"],
        "type": "factual",
        "difficulty": "medium",
    },

    # ── CROSS-REPORT QUERIES (from previous set) ──────────────────────────────

    {
        "id": "Q11",
        "query": "How has GCPL's consolidated revenue evolved from FY 2022-23 to FY 2024-25?",
        "reference_answer": "Consolidated revenue grew from ₹13,315.97 crore in FY 2022-23 to ₹14,364.29 crore in FY 2024-25.",
        "keywords": ["revenue", "consolidated", "13315", "14364", "crore"],
        "relevant_years": ["2022-23", "2024-25"],
        "type": "multi_chunk",
        "difficulty": "hard",   # requires retrieving from 2 different reports
    },
    {
        "id": "Q12",
        "query": "What specific goal did GCPL achieve two years early regarding its 2026 sustainability targets?",
        "reference_answer": "GCPL beat its 2026 goal of 22% plastic reduction two years early, as reported in the FY 2024-25 statement.",
        "keywords": ["22%", "plastic", "2026", "two years early", "sustainability"],
        "relevant_years": ["2024-25"],
        "type": "factual",
        "difficulty": "medium",
    },
    {
        "id": "Q13",
        "query": "What percentage of GCPL's total energy consumption was derived from renewable sources by FY 2024-25?",
        "reference_answer": "As of FY 2024-25, 35% of GCPL's total energy consumption is derived from renewable sources, meeting the company's FY 2025 target.",
        "keywords": ["35%", "renewable", "energy", "FY 2025", "target"],
        "relevant_years": ["2024-25"],
        "type": "factual_numeric",
        "difficulty": "easy",
    },
    {
        "id": "Q14",
        "query": "What was the performance of the Fab liquid detergent brand in its first year?",
        "reference_answer": "The Fab liquid detergent brand crossed ₹150 crore in topline in its first year and reached an annualized revenue run-rate of ₹250 crore by end of FY 2024-25.",
        "keywords": ["Fab", "150 crore", "250 crore", "detergent", "ARR"],
        "relevant_years": ["2024-25"],
        "type": "factual_numeric",
        "difficulty": "easy",
    },
    {
        "id": "Q15",
        "query": "What was the financial impact of the West Africa GTM shift and macro pressures on FY 2024-25 revenue?",
        "reference_answer": "Net sales in the Africa, Middle East, and USA cluster declined by 8% due to macro pressures and a GTM shift in West Africa, though international margins improved.",
        "keywords": ["West Africa", "GTM", "8%", "declined", "Africa", "Middle East"],
        "relevant_years": ["2024-25"],
        "type": "inferential",
        "difficulty": "hard",
    },
    {
        "id": "Q16",
        "query": "Which new categories did GCPL enter through the acquisition of Raymond Consumer Care's business?",
        "reference_answer": "GCPL entered the Deodorants, Perfumes, and Sexual Wellness categories through brands like Park Avenue and KamaSutra.",
        "keywords": ["Raymond", "deodorants", "perfumes", "sexual wellness", "Park Avenue", "KamaSutra"],
        "relevant_years": ["2024-25", "2023-24"],
        "type": "factual",
        "difficulty": "medium",
    },
    {
        "id": "Q17",
        "query": "How is executive compensation linked to ESG performance at GCPL?",
        "reference_answer": "15% of executive compensation for the MD & CEO and other leaders is directly tied to People & Planet goals, including safety records, human rights, and plastic reduction.",
        "keywords": ["15%", "compensation", "ESG", "People", "Planet", "MD", "CEO"],
        "relevant_years": ["2024-25"],
        "type": "inferential",
        "difficulty": "hard",
    },
    {
        "id": "Q18",
        "query": "What is Project EMBED and what is its population coverage target for FY 2025-26?",
        "reference_answer": "Project EMBED is an initiative to eliminate vector-borne diseases (malaria and dengue); it aims to protect 30 million people by FY 2025-26.",
        "keywords": ["EMBED", "malaria", "dengue", "30 million", "vector-borne"],
        "relevant_years": ["2024-25"],
        "type": "factual",
        "difficulty": "medium",
    },
    {
        "id": "Q19",
        "query": "What is GCPL's strategy for rural distribution in India?",
        "reference_answer": "GCPL has focused on expanding rural reach through direct distribution and channel partner networks, targeting villages and smaller towns as part of its India growth strategy.",
        "keywords": ["rural", "distribution", "channel", "reach", "village"],
        "relevant_years": ["2023-24", "2024-25"],
        "type": "inferential",
        "difficulty": "medium",
    },
    {
        "id": "Q20",
        "query": "What digital transformation initiatives has GCPL undertaken across its operations?",
        "reference_answer": "GCPL has invested in data analytics, digital supply chain tools, and technology platforms to drive operational efficiency and consumer insights across its markets.",
        "keywords": ["digital", "technology", "analytics", "data", "transformation"],
        "relevant_years": ["2023-24", "2024-25"],
        "type": "inferential",
        "difficulty": "medium",
    },
]


# ── DIFFICULTY BREAKDOWN (for documentation) ──────────────────────────────────
#
#  Easy   (4): Q01, Q13, Q14  — answer in a single obvious chunk
#  Medium (9): Q02, Q04, Q05, Q06, Q09, Q10, Q12, Q16, Q18, Q19, Q20
#  Hard   (7): Q03, Q07, Q08, Q11, Q15, Q17  — buried stats or multi-report
#
# Type breakdown:
#  factual_numeric : Q03, Q04, Q05, Q07, Q08, Q13, Q14
#  factual         : Q02, Q06, Q09, Q10, Q12, Q16, Q18
#  inferential     : Q01, Q15, Q17, Q19, Q20
#  multi_chunk     : Q11
#
# Report coverage:
#  FY 2020-21 : Q01, Q02
#  FY 2021-22 : Q03, Q04
#  FY 2022-23 : Q05, Q06, Q11
#  FY 2023-24 : Q07, Q08, Q16
#  FY 2024-25 : Q09, Q10, Q12, Q13, Q14, Q15, Q17, Q18, Q19, Q20

In [32]:
evaluation_rows = []

for col, (cdir, label) in CONFIGS.items():
    for test in test_set:
        results  = retrieve(cdir, col.replace("_v2", ""), test["query"])
        precision = precision_at_k(results, test["keywords"])
        recall    = recall_at_k(results,    test["keywords"])
        evaluation_rows.append({
            "collection": col,
            "query_id":   test["id"],
            "query":      test["query"],
            "type":       test["type"],
            "difficulty": test["difficulty"],
            "precision":  round(precision, 3),
            "recall":     round(recall,    3),
        })

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument bu

In [35]:
df = pd.DataFrame(evaluation_rows)
print(df.to_string(index=False))

summary = df.groupby("collection").agg(
    avg_precision=("precision", "mean"),
    avg_recall=("recall",    "mean")
).round(3)

         collection query_id                                                                                                                               query            type difficulty  precision  recall
       fixed__local      Q01                                        What major shift in consumer behavior did GCPL observe across its markets during FY 2020-21?     inferential       easy        0.2   0.167
       fixed__local      Q02   When did GCPL conduct the extensive materiality exercise that formed the basis for its reporting methodology in subsequent years?         factual       hard        0.8   0.400
       fixed__local      Q03       What was the percentage of representation for local ethnic talent in GCPL's Africa senior leadership roles during FY 2021-22? factual_numeric       hard        0.8   1.000
       fixed__local      Q04                  How many financial years had GCPL maintained a status of sending zero waste to landfills by the end of FY 2022-23? factual_num

In [36]:
print("\n--- Summary ---")
print(summary)

df.to_csv("/content/retrieval_benchmark.csv", index=False)
print("\nSaved /content/retrieval_benchmark.csv")


--- Summary ---
                     avg_precision  avg_recall
collection                                    
fixed__local                  0.65       0.490
fixed__local_v2               0.76       0.594
recursive__local              0.72       0.513
recursive__local_v2           0.75       0.608

Saved /content/retrieval_benchmark.csv


In [ ]:
from google.colab import files
files.download('chroma_database.zip')
files.download('chroma_database_v2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Generating the Answers

In [37]:
! pip install transformers accelerate -q

In [38]:
from transformers import pipeline

In [39]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

Device set to use cpu


In [40]:
def generate_answer(query, results):
    # take top 4 retrieved chunks
    context = " ".join(results["documents"][0][:4])
    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context[:1000]}

Question: {query}

Answer:
"""
    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )
    return output[0]["generated_text"]

In [41]:
generation_rows = []

for test in test_set:
    results = retrieve(chroma_dir_v2, "recursive__local", test["query"])
    answer = generate_answer(test["query"], results)
    generation_rows.append({
        "query_id": test["id"],
        "question": test["query"],
        "generated_answer": answer
    })
    print("\n", test["query"])
    print("Answer:", answer)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



 What major shift in consumer behavior did GCPL observe across its markets during FY 2020-21?
Answer: shared services, and brought sharper cost formats. global brands from day one. This shift is not control across the board. just operational, it is cultural. It reflects our We have now established GCPL International belief that GCPL can be a consumer products Operations—a focused team that looks company with ideas and impact that travel at distribution, innovation, marketing and across the world. execution across 80+ export countries, with a highly lean centralised set-up. We are scaling up our global blockbuster portfolio 50 Your media and creative approach seems to have fundamentally shifted. What is driving this change? years, with the ambition to make it stronger and more purposeful. GCPL’s purpose to


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



 When did GCPL conduct the extensive materiality exercise that formed the basis for its reporting methodology in subsequent years?
Answer: In fiscal year 2019-20, we materiality matrix. Primary inputs were We approach materiality from a strategic The process of determining materiality Materiality analysis was performed and value creation perspective. Material at GCPL is compliant with the prescriptions through identification and prioritisation. A issues are identified by engaging in prescriptions of the IIRC and systematic step-wise process was conversations with our stakeholders and draws from the six capitals of followed. First, relevant insights were monitoring broad trends in the industry. integrated reporting. collected through with primary and Performance on material issues forms the secondary research, and then, necessary core content of this Annual and Integrated calculations were performed to obtain the Report. In fiscal year 2019-20, we material


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


...


 How is executive compensation linked to ESG performance at GCPL?
Answer: For renewables, goals for executive compensation include to increase renewable energy use, decrease specific energy and reduce emissions 20 Material issues of our external stakeholders Product Safety and Quality Testing Sustainable Packaging Type of impact Type of impact Negative Positive Business Case Business Case We understand the failure to adapt to ever evolving GCPL is proactively adapting best practices in • OHS training coverage – 100% Link to executive At GCPL, the executive compensation of all leaders comprises of 15% of people & planet goals. They compensation are in line with the company’s vision to foster an inspiring workplace and build an equitable and greener planet. For renewables, goals for executive compensation include to increase renewable energy use, decrease specific energy and reduce emissions 20 Material issues of


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



 What is Project EMBED and what is its population coverage target for FY 2025-26?
Answer: Protecting people from vector-borne diseases Project EMBED was started in 2015 in We worked in each location for 3 years, In the fiscal year 2020-21, we initiated Project EMBED was started in 2015 in We worked in each location for 3 years, In the fiscal year 2020-21, we initiated Madhya Pradesh in partnership with the spreading awareness among households interventions on dengue and chikungunya Ministry of Health and Family Welfare’s and people at the bottom of the pyramid prevention in urban areas of the National Centre for Vector-borne and vulnerable and marginalised gro


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



 What is GCPL's strategy for rural distribution in India?
Answer: reducing in Madhya Pradesh. Lucknow, program and  4.92 worth waste across all its plants, Kanpur, Agra, Meerut, Firozabad of value for the urban processes, products, and supply and Prayagraj in Uttar Pradesh.

 What digital transformation initiatives has GCPL undertaken across its operations?
Answer: We have now established GCPL International belief that GCPL can be a consumer products Operations—a focused team that looks company with ideas and impact that travel at distribution, innovation, marketing and across the world. execution across 80+ export countries, with a highly lean centralised set-up. We are scaling up our global blockbuster portfolio 50 Your media and creative approach seems to have fundamentally shifted. What is driving this change? has already doubled our Rural footprint. The outsourcing direct operations to distributors. resultant scale-up in in Direct Distribution This strategic move has significantl

In [46]:
ref_map = {t["query"]: t["reference_answer"] for t in test_set}

gen_df = pd.DataFrame(generation_rows)
gen_df["qual_score"] = gen_df.apply(
    lambda row: qualitative_score(row["generated_answer"], ref_map[row["question"]]),
    axis=1
)
print("Avg qualitative score:", round(gen_df["qual_score"].mean(), 2), "/ 2")
print(gen_df[["query_id", "qual_score"]].to_string(index=False))
gen_df.to_csv("/content/generated_answers.csv", index=False)
print("Saved /content/generated_answers.csv")

Avg qualitative score: 0.85 / 2
query_id  qual_score
     Q01           2
     Q02           1
     Q03           1
     Q04           0
     Q05           2
     Q06           2
     Q07           0
     Q08           1
     Q09           1
     Q10           0
     Q11           1
     Q12           0
     Q13           0
     Q14           2
     Q15           0
     Q16           0
     Q17           1
     Q18           2
     Q19           0
     Q20           1
Saved /content/generated_answers.csv


In [47]:
from google.colab import files
files.download("/content/generated_answers.csv")
files.download("/content/retrieval_benchmark.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>